# Endpoint singularities with tanh–sinh quadrature
Estimate the integral of x raised to minus alpha on [0,1]. The integrand is unbounded, yet its integral exists for alpha below one. Compare the physical midpoint grid with a double-exponential change of variable. All controls and calculations run with NumPy and Matplotlib.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

alpha = 0.65
cutoff = 3.0                  # half-width in transformed coordinates
step = 0.25
assert 0.05 <= alpha <= 0.95 and 1 <= cutoff <= 6 and .04 <= step <= .5

def transformed(t, exponent):
    s = np.pi*np.sinh(t)
    log_x = -np.logaddexp(0., -s)
    log_1mx = -np.logaddexp(0., s)
    log_values = np.log(np.pi) + np.log(np.cosh(t)) + (1-exponent)*log_x + log_1mx
    return np.exp(log_values)

def tanh_sinh(T, h, exponent):
    n = int(np.ceil(T/h))
    t = np.linspace(-T, T, 2*n+1)
    g = transformed(t, exponent)
    # Composite trapezoid on the truncated interval.
    value = (T/n)*(g.sum()-(g[0]+g[-1])/2)
    return value, len(t)

def midpoint(n, exponent):
    return np.mean(((np.arange(n)+.5)/n)**(-exponent))

def exact_truncation(T, exponent):
    # Left and right tails calculated without subtracting nearly equal numbers.
    s = np.pi*np.sinh(T)
    log_left = -np.logaddexp(0., s)
    log_right = -np.logaddexp(0., -s)
    return (np.exp((1-exponent)*log_left)-np.expm1((1-exponent)*log_right))/(1-exponent)

exact = 1/(1-alpha)
value, nodes = tanh_sinh(cutoff,step,alpha)
print(f"Exact {exact:.10f}; tanh–sinh {value:.10f} with {nodes} nodes")
print("Omitted tail alone:", exact_truncation(cutoff,alpha))

## Plot the transformed integrand and separate both errors
The midpoint rule in the original coordinate never evaluates zero. The transformed quadrature additionally concentrates nodes where the original slope diverges. Reducing the transformed step at fixed cutoff eventually leaves the omitted tail exposed.

In [ ]:
fig, (shape, errors) = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
tt = np.linspace(-5, 4, 450)
for exponent, color in [(alpha, "#145c71"),(.85,"#b94626")]:
    shape.plot(tt, transformed(tt,exponent), color=color, label=f"α = {exponent:g}")
shape.axvspan(-cutoff,cutoff,color="grey",alpha=.08)
shape.set(xlabel="Transformed coordinate t",ylabel="Weight times integrand",title="Where does the integral live?")
shape.legend()
counts = 2*np.array([4,6,8,12,16,24,32,48,64])+1
de = [abs(tanh_sinh(cutoff,cutoff/(n//2),alpha)[0]-exact) for n in counts]
errors.loglog(counts,np.maximum(de,1e-16),"o-",color="#145c71",label="tanh–sinh")
errors.loglog(counts,[abs(midpoint(n,alpha)-exact) for n in counts],"s--",color="#b94626",label="midpoint in x")
errors.axhline(max(1e-16,exact_truncation(cutoff,alpha)),ls=":",color="black",label="truncation alone")
errors.set(xlabel="Function evaluations",ylabel="Absolute error",title="Refinement at a fixed cutoff")
errors.legend(fontsize=9)
for ax in (shape,errors): ax.grid(alpha=.2)
plt.show()

## Check a finite integral independently
A second substitution x = u raised to the power 1/(1−alpha) makes the integrand exactly constant. Try alpha 0.95 before increasing the cutoff. More nodes at fixed cutoff cannot replace the missing area beyond the window.

In [ ]:
exponent = 1/(1-alpha)
u = np.linspace(.1,.99,10)
flat = exponent*u**(exponent-1-alpha*exponent)
assert np.allclose(flat,exact)
test, _ = tanh_sinh(5.5,.075,.65)
assert abs(test-1/.35) < 5e-13
print("Analytical substitution and converged reference checks passed.")

## Check the retained window with an element quadrature
The exact finite-window target equals the full integral minus the two omitted tails. A local OneSimulation installation can measure this finite part by building a physical line mesh in the transformed variable and integrating at mapped Gaussian quadrature points of each element. This is a different sampling grid from the trapezoid. The analytical finite-window check runs in every environment.

In [ ]:
finite_target = exact - exact_truncation(cutoff,alpha)
print(f"Window loss {exact-finite_target:.8g}; signed panel error {finite_target-value:.8g}")
try:
    from onesim.cad.d1 import Line
    from onesim.meshing import StructuredLineMesher
    from onesim.numerics.fem.backends.spaces.line_lagrange import LineLagrangeSpace
except ImportError:
    print("The mapped Gaussian element comparison is available with OneSimulation installed locally.")
else:
    line = Line(start=(-cutoff,0,0),end=(cutoff,0,0))
    mesh = StructuredLineMesher(cells=24).generate(line)
    space = LineLagrangeSpace(mesh,order=1,quadrature_order=8)
    finite_gauss = 0.
    for i in range(len(space.cells)):
        points,weights = space.quadrature(i)
        finite_gauss += weights @ transformed(points[:,0],alpha)
    assert np.isclose(finite_gauss,finite_target,atol=2e-10,rtol=1e-11)
    print(f"OneSimulation finite-window integral {finite_gauss:.10f}.")